### 1. Import Library dan Konfigurasi Pengujian Clustering
Sel ini bertugas mengimpor library algoritma clustering (K-Means, Agglomerative, DBSCAN), metrik evaluasi internal dan eksternal, t-SNE, serta modul visualisasi grafik.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans                 
from sklearn.cluster import AgglomerativeClustering 
from sklearn.cluster import DBSCAN                  
from sklearn.neighbors import NearestNeighbors      
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, confusion_matrix, classification_report
from scipy.cluster.hierarchy import dendrogram, linkage 
from scipy.optimize import linear_sum_assignment       
from sklearn.manifold import TSNE  
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi gaya tampilan grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110


### 2. Setup Path dan Pemuatan Matriks Fitur PCA (CSV)
Sel ini bertugas menentukan lokasi folder dan memuat matriks fitur 50 dimensi dari file `output_data/fitur_pca.csv`.


In [ ]:
CURRENT_DIR = Path.cwd().resolve()
if CURRENT_DIR.name == "kode_program":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

OUTPUT_DIR = PROJECT_ROOT / "output_data"
CSV_FITUR_PATH = OUTPUT_DIR / "fitur_pca.csv"

# Verifikasi ketersediaan file fitur hasil Tahap 2
if not CSV_FITUR_PATH.exists():
    raise FileNotFoundError(f"File {CSV_FITUR_PATH} belum ditemukan! Jalankan '02_ekstraksi_fitur.ipynb' terlebih dahulu.")

print(f"Memuat data fitur langsung dari: {CSV_FITUR_PATH}")
df_fitur = pd.read_csv(CSV_FITUR_PATH)

# 1. Memisahkan Kolom Fitur Numerik PCA (pca_1 sampai pca_50) ke dalam Matriks X
pca_cols = [c for c in df_fitur.columns if c.startswith("pca_")]
X = df_fitur[pca_cols].values

# 2. Mengambil Label Motif Asli (Ground Truth) untuk Kebutuhan Validasi
y_true_names = df_fitur["motif_utama"].values
df_ukiran = df_fitur[["nama_file", "motif_utama"]]

print("Data Berhasil Dimuat:")
print(f"- Matriks Fitur PCA (X) : Shape = {X.shape} (2.364 citra x 50 fitur)")
print(f"- Total Citra Terdaftar : {len(y_true_names)} entri")

# 3. Menyiapkan Pemetaan ID Angka untuk Evaluasi Label Asli
unique_motifs = sorted(list(set(y_true_names)))
label_to_id = {name: i for i, name in enumerate(unique_motifs)}
y_true = np.array([label_to_id[name] for name in y_true_names])

print(f"- Daftar Motif Asli     : {unique_motifs} (K = {len(unique_motifs)})")


### 3. Pencarian Parameter Optimal K-Means (Elbow Method & Silhouette Score)
Sel ini bertugas menguji variasi jumlah klaster $k \in [2, 10]$ untuk melihat titik siku (Elbow) dan nilai Silhouette Score tertinggi.


In [ ]:
# Menguji Rentang Nilai k dari 2 sampai 10
k_range = list(range(2, 11))
inertias = []
sil_scores_kmeans = []

# Perulangan untuk melatih K-Means pada setiap nilai k
for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X)
    
    # Simpan nilai Inertia (Within-Cluster Sum of Squares) dan Silhouette Score
    inertias.append(km.inertia_)
    sil_scores_kmeans.append(silhouette_score(X, labels))

# Visualisasi Dua Sumbu (Twin Axis): Inertia di Kiri dan Silhouette di Kanan
fig, ax1 = plt.subplots(figsize=(11, 4.5))

# Sumbu Kiri: Kurva Elbow (Inertia)
color = 'tab:blue'
ax1.set_xlabel('Jumlah Klaster (k)', fontsize=12, labelpad=10)
ax1.set_ylabel('Inertia / WCSS (Within-Cluster Sum of Squares)', color=color, fontsize=12)
line1 = ax1.plot(k_range, inertias, marker='o', linewidth=2, color=color, label='Inertia (Elbow Method)')
ax1.tick_params(axis='y', labelcolor=color)

# Sumbu Kanan: Kurva Silhouette Score
ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('Rata-rata Silhouette Score', color=color, fontsize=12)
line2 = ax2.plot(k_range, sil_scores_kmeans, marker='s', linewidth=2, linestyle='--', color=color, label='Silhouette Score')
ax2.tick_params(axis='y', labelcolor=color)

# Garis vertikal penanda k=5 (Jumlah kelas motif asli ukiran Bali)
ax1.axvline(x=5, color='crimson', linestyle=':', linewidth=1.8, label='k=5 (Jumlah Motif Alami)')

# Menggabungkan legenda dari kedua sumbu
lines = line1 + line2 + [ax1.get_lines()[-1]]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='center right')

plt.title('Pencarian Parameter Optimal K-Means (Elbow Method & Silhouette Curve)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


### 4. Visualisasi Pohon Dendrogram Hierarchical Clustering (Ward Linkage)
Sel ini bertugas menghitung matriks kedekatan hierarkis dan menampilkan pohon dendrogram dengan garis potong batas 5 klaster utama.


In [ ]:
# 1. Menghitung Matriks Linkage Ward (Meminimalkan Variansi di dalam Klaster)
Z = linkage(X, method='ward')

plt.figure(figsize=(12, 5.5))

# 2. Menggambar Pohon Dendrogram dengan Tampilan Truncated (p=25) agar Tidak Terlalu Rapat
dend = dendrogram(
    Z,
    truncate_mode='lastp',
    p=25,
    leaf_rotation=45.,
    leaf_font_size=10.,
    show_contracted=True
)

# 3. Menghitung dan Menggambar Garis Cut-off Threshold untuk Memisahkan 5 Klaster
cutoff_distance = (Z[-4, 2] + Z[-5, 2]) / 2
plt.axhline(y=cutoff_distance, color='crimson', linestyle='--', linewidth=1.5, 
            label=f'Cut-off Threshold (5 Klaster Utama, Jarak ≈ {cutoff_distance:.2f})')

plt.title('Pohon Dendrogram Hierarchical Clustering (Ward Linkage - Truncated Display)', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Indeks Klaster / Jumlah Sampel per Daun', fontsize=11, labelpad=8)
plt.ylabel('Jarak Euclidean Linkage', fontsize=11, labelpad=8)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


### 5. Penentuan Parameter Radius eps DBSCAN (K-Distance Graph)
Sel ini bertugas menghitung jarak ke tetangga ke-5 terdekat dan memplot kurva K-Distance untuk menemukan nilai ambang batas radius `eps` optimal.


In [ ]:
# 1. Menghitung Jarak ke Tetangga Terdekat ke-5 (k = 5) untuk Setiap Titik Data
k_neighbors = 5
nbrs = NearestNeighbors(n_neighbors=k_neighbors).fit(X)
distances, indices = nbrs.kneighbors(X)

# 2. Mengambil Jarak ke Tetangga Terjauh ke-5 lalu Mengurutkannya dari Terkecil ke Terbesar
k_distances = np.sort(distances[:, k_neighbors - 1])

# 3. Mengestimasi Titik Siku (Knee Point) pada Persentil 90%
knee_idx = int(len(k_distances) * 0.90)
eps_optimal = float(k_distances[knee_idx])

# 4. Membuat Plot Kurva K-Distance
plt.figure(figsize=(10, 4.8))
plt.plot(k_distances, color='purple', linewidth=2, label='Jarak ke Tetangga ke-5')
plt.axhline(y=eps_optimal, color='crimson', linestyle='--', linewidth=1.5, label=f'Rekomendasi eps ≈ {eps_optimal:.3f}')
plt.scatter([knee_idx], [eps_optimal], color='red', s=60, zorder=5)

plt.annotate(
    f"Knee Point: eps = {eps_optimal:.3f}",
    (knee_idx, eps_optimal),
    textcoords="offset points", xytext=(-100, 30),
    bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.7),
    arrowprops=dict(arrowstyle="->", color="black"),
    fontweight="bold"
)

plt.title('K-Distance Graph untuk Penentuan Parameter Radius eps DBSCAN', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Indeks Titik Data Terurut (Berdasarkan Jarak)', fontsize=11, labelpad=8)
plt.ylabel(f'Jarak ke Tetangga Terdekat ke-{k_neighbors}', fontsize=11, labelpad=8)
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


### 6. Pelatihan Ketiga Algoritma dan Tabel Evaluasi Metrik Internal
Sel ini bertugas melatih model K-Means, Tree (Agglomerative), dan DBSCAN, lalu menampilkan tabel perbandingan metrik kualitas klaster tanpa label (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index).


In [ ]:
# 1. Melatih Algoritma K-Means (k = 5)
kmeans_model = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
labels_kmeans = kmeans_model.fit_predict(X)

# 2. Melatih Algoritma Tree / Agglomerative Hierarchical Clustering (k = 5, Ward Linkage)
tree_model = AgglomerativeClustering(n_clusters=5, linkage='ward')
labels_tree = tree_model.fit_predict(X)

# 3. Melatih Algoritma DBSCAN (Radius eps hasil K-Distance, min_samples = 5)
min_samples_val = 5
dbscan_model = DBSCAN(eps=eps_optimal, min_samples=min_samples_val)
labels_dbscan = dbscan_model.fit_predict(X)

def get_internal_metrics(X_data, labels):
    """
    Fungsi untuk menghitung metrik evaluasi internal murni tanpa label:
    - Silhouette Score (semakin mendekati 1 semakin baik)
    - Davies-Bouldin Index (semakin mendekati 0 semakin baik)
    - Calinski-Harabasz Index (semakin tinggi semakin baik)
    Titik noise (-1) pada DBSCAN difilter agar tidak merusak perhitungan jarak klaster.
    """
    unique_lbls = set(labels)
    mask = labels != -1 # Menyaring titik noise
    n_clusters = len(unique_lbls - {-1})
    
    if n_clusters < 2:
        return np.nan, np.nan, np.nan
        
    sil = silhouette_score(X_data[mask], labels[mask])
    dbi = davies_bouldin_score(X_data[mask], labels[mask])
    chi = calinski_harabasz_score(X_data[mask], labels[mask])
    return sil, dbi, chi

# Menghitung skor metrik internal untuk ketiga algoritma
sil_km, dbi_km, chi_km = get_internal_metrics(X, labels_kmeans)
sil_tr, dbi_tr, chi_tr = get_internal_metrics(X, labels_tree)
sil_db, dbi_db, chi_db = get_internal_metrics(X, labels_dbscan)

# 4. Membuat Tabel Ringkasan Evaluasi Metrik Internal
df_internal = pd.DataFrame({
    "Algoritma Clustering": [
        "K-Means (Partition-based)",
        "Tree / Agglomerative (Connectivity-based)",
        "DBSCAN (Density-based)"
    ],
    "Jumlah Klaster": [
        len(set(labels_kmeans)),
        len(set(labels_tree)),
        len(set(labels_dbscan) - {-1})
    ],
    "Silhouette Score (↑ Lebih Tinggi Lebih Baik)": [
        f"{sil_km:.4f}",
        f"{sil_tr:.4f}",
        f"{sil_db:.4f}" if not np.isnan(sil_db) else "N/A"
    ],
    "Davies-Bouldin Index (↓ Lebih Rendah Lebih Baik)": [
        f"{dbi_km:.4f}",
        f"{dbi_tr:.4f}",
        f"{dbi_db:.4f}" if not np.isnan(dbi_db) else "N/A"
    ],
    "Calinski-Harabasz Index (↑ Lebih Tinggi Lebih Baik)": [
        f"{chi_km:.1f}",
        f"{chi_tr:.1f}",
        f"{chi_db:.1f}" if not np.isnan(chi_db) else "N/A"
    ]
})

print("=== TABEL PENGUJIAN METRIK INTERNAL (KUALITAS KLASTER MURNI TANPA LABEL) ===")
display(df_internal)


### 7. Pengujian Akurasi Terhadap Label Asli Dataset (Hungarian Algorithm & Classification Report)
Sel ini bertugas memetakan nomor klaster ke nama motif asli secara optimal, menghitung Clustering Accuracy/Purity (%), ARI, NMI, serta mencetak laporan klasifikasi per motif ukiran.


In [ ]:
def map_clusters_to_motif_names(y_true_arr, y_pred_clusters):
    """
    Fungsi untuk memetakan nomor klaster tanpa label (0, 1, 2, dll) ke nama motif asli.
    - Untuk K-Means & Tree (k=5): Menggunakan Hungarian Algorithm (linear_sum_assignment)
      untuk mencari pasangan 1-ke-1 yang menghasilkan akurasi pencocokan tertinggi.
    - Untuk DBSCAN: Memetakan setiap klaster mikro ke motif mayoritas di dalamnya.
    - Titik noise (-1) otomatis diberi label 'Noise / Outlier'.
    """
    valid_mask = (y_pred_clusters != -1)
    if valid_mask.sum() == 0:
        return np.array(['Noise / Outlier'] * len(y_pred_clusters)), {}
        
    # Membuat tabel kontingensi frekuensi kemunculan motif per klaster
    df_cross = pd.crosstab(
        pd.Series(y_pred_clusters[valid_mask], name='cluster'),
        pd.Series(y_true_arr[valid_mask], name='motif')
    )
    
    cluster_mapping = {}
    # Jika jumlah klaster sama dengan jumlah motif asli (5 klaster), gunakan Hungarian Algorithm
    if df_cross.shape[0] == df_cross.shape[1] and df_cross.shape[1] > 1:
        cost_matrix = -df_cross.values
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        cluster_ids = df_cross.index.values
        motif_names = df_cross.columns.values
        for r, c in zip(row_ind, col_ind):
            cluster_mapping[cluster_ids[r]] = motif_names[c]
    else:
        # Jika jumlah klaster berbeda (seperti DBSCAN), petakan ke motif mayoritas
        for cid in df_cross.index:
            dominant_motif = df_cross.loc[cid].idxmax()
            cluster_mapping[cid] = dominant_motif
            
    # Mengonversi array nomor klaster ke array nama motif hasil prediksi
    y_pred_names = []
    for c in y_pred_clusters:
        if c == -1:
            y_pred_names.append('Noise / Outlier')
        elif c in cluster_mapping:
            y_pred_names.append(cluster_mapping[c])
        else:
            y_pred_names.append('Lainnya')
            
    return np.array(y_pred_names), cluster_mapping

# 1. Menjalankan Pemetaan Nama Motif untuk Ketiga Algoritma
y_pred_km_named, map_km = map_clusters_to_motif_names(y_true_names, labels_kmeans)
y_pred_tr_named, map_tr = map_clusters_to_motif_names(y_true_names, labels_tree)
y_pred_db_named, map_db = map_clusters_to_motif_names(y_true_names, labels_dbscan)

print('=== KAMUS PEMETAAN KLASTER KE MOTIF ASLI ===')
print('K-Means Mapping :', {int(k): v for k, v in map_km.items()})
print('Tree Mapping    :', {int(k): v for k, v in map_tr.items()})
print(f'DBSCAN Mapping  : Berhasil memetakan {len(map_db)} klaster mikro ke motif dominan.')

def compute_external_scores(y_true_raw, y_pred_raw, y_pred_named):
    """
    Menghitung skor evaluasi eksternal terhadap label motif asli:
    - Purity / Accuracy (%): Persentase tebakan motif yang benar
    - ARI (Adjusted Rand Index): Kesamaan pasangan klaster terkoreksi peluang acak
    - NMI (Normalized Mutual Information): Informasi timbal-balik yang ditangkap
    - Noise (%): Persentase data yang diklasifikasikan sebagai noise (-1)
    """
    is_noise = (y_pred_raw == -1)
    noise_pct = (np.sum(is_noise) / len(y_pred_raw)) * 100
    
    purity = (y_true_raw == y_pred_named).sum() / len(y_true_raw) * 100
    ari = adjusted_rand_score(y_true_raw, y_pred_raw)
    nmi = normalized_mutual_info_score(y_true_raw, y_pred_raw)
    n_clusters = len(set(y_pred_raw) - {-1})
    
    return purity, ari, nmi, noise_pct, n_clusters

# Menghitung skor eksternal untuk ketiga algoritma
purity_km, ari_km, nmi_km, noise_km, n_km = compute_external_scores(y_true_names, labels_kmeans, y_pred_km_named)
purity_tr, ari_tr, nmi_tr, noise_tr, n_tr = compute_external_scores(y_true_names, labels_tree, y_pred_tr_named)
purity_db, ari_db, nmi_db, noise_db, n_db = compute_external_scores(y_true_names, labels_dbscan, y_pred_db_named)

# 2. Menampilkan Tabel Ringkasan Akurasi Pengujian
df_external = pd.DataFrame({
    'Algoritma Clustering': [
        'K-Means (Partition-based)',
        'Tree / Agglomerative (Ward Linkage)',
        'DBSCAN (Density-based)'
    ],
    'Jumlah Klaster': [n_km, n_tr, n_db],
    'Noise (%)': [f'{noise_km:.2f}%', f'{noise_tr:.2f}%', f'{noise_db:.2f}%'],
    'Clustering Accuracy / Purity (%)': [f'{purity_km:.2f}%', f'{purity_tr:.2f}%', f'{purity_db:.2f}%'],
    'Adjusted Rand Index (ARI)': [f'{ari_km:.4f}', f'{ari_tr:.4f}', f'{ari_db:.4f}'],
    'Normalized Mutual Info (NMI)': [f'{nmi_km:.4f}', f'{nmi_tr:.4f}', f'{nmi_db:.4f}']
})

print('\n=== TABEL PERBANDINGAN HASIL PENGUJIAN TERHADAP LABEL ASLI ===')
display(df_external)

# 3. Menampilkan Laporan Klasifikasi Rinci (Precision, Recall, F1-Score) per Motif
print('\n=== DETAIL KINERJA PER MOTIF (K-MEANS) ===')
print(classification_report(y_true_names, y_pred_km_named, zero_division=0))

print('\n=== DETAIL KINERJA PER MOTIF (TREE / AGGLOMERATIVE) ===')
print(classification_report(y_true_names, y_pred_tr_named, zero_division=0))


### 8. Visualisasi Confusion Matrix Pemetaan Motif untuk Ketiga Algoritma
Sel ini bertugas menampilkan grafik matriks perbandingan antara Label Motif Asli vs Motif Hasil Prediksi Klaster untuk K-Means, Tree, dan DBSCAN.


In [ ]:
# 1. Menyiapkan Kanvas 1 Baris x 3 Kolom untuk Heatmap Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

algo_mappings = [
    ('K-Means', y_pred_km_named, axes[0]),
    ('Tree (Agglomerative)', y_pred_tr_named, axes[1]),
    ('DBSCAN', y_pred_db_named, axes[2])
]

# 2. Menggambar Heatmap Confusion Matrix untuk Setiap Algoritma
for title, pred_names, ax in algo_mappings:
    all_col_labels = unique_motifs.copy()
    if 'Noise / Outlier' in pred_names:
        all_col_labels.append('Noise / Outlier')
        
    # Menghitung matriks kontingensi antara motif asli dan motif prediksi
    cm = confusion_matrix(y_true_names, pred_names, labels=all_col_labels)
    cm_display = cm[:len(unique_motifs), :] # Menampilkan baris motif asli
    
    # Visualisasi Heatmap dengan Seaborn
    sns.heatmap(
        cm_display,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=all_col_labels,
        yticklabels=unique_motifs,
        cbar=True,
        ax=ax,
        linewidths=0.5
    )
    ax.set_title(f'Confusion Matrix: {title}', fontsize=12, fontweight='bold', pad=10)
    ax.set_ylabel('Label Motif Asli (Ground Truth)', fontsize=11)
    ax.set_xlabel('Motif Hasil Prediksi Klaster', fontsize=11)
    ax.tick_params(axis='x', rotation=25)

plt.suptitle('Confusion Matrix Pencocokan Klaster Terhadap Label Motif Ukiran Bali', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()


### 9. Visualisasi Sebaran Titik Klaster 2D (Proyeksi t-SNE)
Sel ini bertugas memproyeksikan fitur ke ruang 2 dimensi menggunakan t-SNE dan menampilkan 4 grafik sebaran titik berdampingan (Motif Asli vs K-Means vs Tree vs DBSCAN).


In [ ]:
print("Menghitung proyeksi t-SNE 2D (Mereduksi 50 Fitur PCA ke 2 Dimensi)...")
# t-SNE memproyeksikan manifold non-linear ke dalam bidang datar 2D untuk visualisasi visual
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X)

# Membuat Kanvas 1 Baris x 4 Kolom
fig, axes = plt.subplots(1, 4, figsize=(24, 5.5))

# 1. Subplot Label Motif Asli (Ground Truth)
scatter1 = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true, cmap='tab10', s=12, alpha=0.7)
axes[0].set_title("Label Motif Asli (Ground Truth)", fontsize=13, fontweight='bold')
handles, _ = scatter1.legend_elements()
axes[0].legend(handles, unique_motifs, loc='upper right', title="Motif", fontsize=9)

# 2. Subplot Hasil Klaster K-Means
scatter2 = axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_kmeans, cmap='tab10', s=12, alpha=0.7)
axes[1].set_title(f"K-Means (k=5)\nPurity: {purity_km:.1f}%, ARI: {ari_km:.3f}", fontsize=13, fontweight='bold')

# 3. Subplot Hasil Klaster Tree (Agglomerative)
scatter3 = axes[2].scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels_tree, cmap='tab10', s=12, alpha=0.7)
axes[2].set_title(f"Tree / Agglomerative (Ward)\nPurity: {purity_tr:.1f}%, ARI: {ari_tr:.3f}", fontsize=13, fontweight='bold')

# 4. Subplot Hasil Klaster DBSCAN (Titik Noise Ditandai Warna Abu-Abu Gelap)
db_colors = ['dimgray' if l == -1 else plt.cm.tab10(l % 10) for l in labels_dbscan]
axes[3].scatter(X_tsne[:, 0], X_tsne[:, 1], c=db_colors, s=12, alpha=0.7)
axes[3].set_title(f"DBSCAN (eps={eps_optimal:.2f})\nKlaster: {n_db}, Noise: {noise_db:.1f}%", fontsize=13, fontweight='bold')

# Merapikan Sumbu Koordinat
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("t-SNE Dimensi 1", fontsize=10)
    ax.set_ylabel("t-SNE Dimensi 2", fontsize=10)

plt.suptitle("Perbandingan Sebaran Klaster 2D (t-SNE): Ground Truth vs K-Means vs Tree vs DBSCAN", fontsize=15, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()
